In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\Machine Learning\upi_fraud_detection\data\processed\feature_engineered_transactions.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df[["transaction_id", "sender_id", "timestamp", "amount"]].head())

Shape: (15000, 56)

Columns:
['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'amount', 'merchant_category', 'transaction_type', 'city', 'latitude', 'longitude', 'device_id', 'upi_channel', 'fraud_label', 'fraud_scenario', 'date', 'hour', 'day_of_week', 'is_weekend', 'is_late_night', 'user_id', 'profile_type', 'avg_transaction_amount', 'daily_transaction_rate', 'home_city', 'home_lat', 'home_lon', 'registered_device', 'amount_vs_user_avg', 'amount_deviation', 'amount_ratio', 'is_new_device', 'distance_from_home_km', 'transactions_last_10m', 'transactions_last_1h', 'transactions_last_24h', 'time_since_previous_transaction', 'recipient_transaction_count', 'is_new_recipient', 'device_transaction_count', 'location_transaction_count', 'is_new_location', 'distance_from_previous_location_km', 'travel_speed_kmph', 'impossible_travel_flag', 'amount_vs_user_std', 'extreme_amount_flag', 'historical_avg_amount', 'historical_amount_std', 'historical_amount_zscore', 'daily_spending', 'dai

In [3]:
# Step 1 — Daily Time-Series Dataset

In [4]:
# Sort transactions by user and time
df = df.sort_values(["sender_id", "timestamp"]).copy()

# Create date from timestamp
df["date"] = df["timestamp"].dt.date

# Create daily time-series data
daily = (
    df.groupby(["sender_id", "date"], as_index=False)
      .agg(
          daily_spending=("amount", "sum"),
          daily_transaction_count=("transaction_id", "count")
      )
)

print("Daily dataset shape:", daily.shape)
print(daily.head(10))

Daily dataset shape: (13623, 4)
  sender_id        date  daily_spending  daily_transaction_count
0   U100000  2026-01-03         2582.92                        1
1   U100000  2026-01-05         1327.52                        1
2   U100000  2026-01-14         1007.41                        1
3   U100000  2026-01-15          663.43                        1
4   U100000  2026-01-16          800.96                        1
5   U100000  2026-01-22         1036.17                        1
6   U100000  2026-01-24         1859.02                        1
7   U100000  2026-01-29          562.08                        1
8   U100000  2026-01-31         2023.22                        2
9   U100000  2026-02-27         3669.65                        1


In [5]:
#  7-Day Rolling Spending Baseline
# Sort by user and date
daily = daily.sort_values(["sender_id", "date"]).copy()

# Convert date to datetime
daily["date"] = pd.to_datetime(daily["date"])

# Previous 7 days' average spending
daily["rolling_spending_mean"] = (
    daily.groupby("sender_id")["daily_spending"]
         .transform(
             lambda x: x.shift(1).rolling(window=7, min_periods=3).mean()
         )
)

# Previous 7 days' spending standard deviation
daily["rolling_spending_std"] = (
    daily.groupby("sender_id")["daily_spending"]
         .transform(
             lambda x: x.shift(1).rolling(window=7, min_periods=3).std()
         )
)

print(daily[
    [
        "sender_id",
        "date",
        "daily_spending",
        "rolling_spending_mean",
        "rolling_spending_std"
    ]
].head(15))

   sender_id       date  daily_spending  rolling_spending_mean  \
0    U100000 2026-01-03         2582.92                    NaN   
1    U100000 2026-01-05         1327.52                    NaN   
2    U100000 2026-01-14         1007.41                    NaN   
3    U100000 2026-01-15          663.43            1639.283333   
4    U100000 2026-01-16          800.96            1395.320000   
5    U100000 2026-01-22         1036.17            1276.448000   
6    U100000 2026-01-24         1859.02            1236.401667   
7    U100000 2026-01-29          562.08            1325.347143   
8    U100000 2026-01-31         2023.22            1036.655714   
9    U100000 2026-02-27         3669.65            1136.041429   
10   U100000 2026-03-02          529.85            1516.361429   
11   U100000 2026-03-05         1006.66            1497.278571   
12   U100000 2026-03-16         1608.97            1526.664286   
13   U100001 2026-01-02          731.56                    NaN   
14   U1000

In [6]:
daily["spending_time_zscore"] = (
    (daily["daily_spending"] - daily["rolling_spending_mean"])
    / daily["rolling_spending_std"]
)

# Infinite values ko NaN karo
daily["spending_time_zscore"] = (
    daily["spending_time_zscore"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    daily[
        [
            "sender_id",
            "date",
            "daily_spending",
            "rolling_spending_mean",
            "rolling_spending_std",
            "spending_time_zscore"
        ]
    ].head(15)
)

   sender_id       date  daily_spending  rolling_spending_mean  \
0    U100000 2026-01-03         2582.92                    NaN   
1    U100000 2026-01-05         1327.52                    NaN   
2    U100000 2026-01-14         1007.41                    NaN   
3    U100000 2026-01-15          663.43            1639.283333   
4    U100000 2026-01-16          800.96            1395.320000   
5    U100000 2026-01-22         1036.17            1276.448000   
6    U100000 2026-01-24         1859.02            1236.401667   
7    U100000 2026-01-29          562.08            1325.347143   
8    U100000 2026-01-31         2023.22            1036.655714   
9    U100000 2026-02-27         3669.65            1136.041429   
10   U100000 2026-03-02          529.85            1516.361429   
11   U100000 2026-03-05         1006.66            1497.278571   
12   U100000 2026-03-16         1608.97            1526.664286   
13   U100001 2026-01-02          731.56                    NaN   
14   U1000

In [7]:
# Previous 7 observations ka transaction frequency baseline
daily["rolling_transaction_mean"] = (
    daily.groupby("sender_id")["daily_transaction_count"]
         .transform(
             lambda x: x.shift(1).rolling(window=7, min_periods=3).mean()
         )
)

daily["rolling_transaction_std"] = (
    daily.groupby("sender_id")["daily_transaction_count"]
         .transform(
             lambda x: x.shift(1).rolling(window=7, min_periods=3).std()
         )
)

# Frequency z-score
daily["frequency_time_zscore"] = (
    (daily["daily_transaction_count"] - daily["rolling_transaction_mean"])
    / daily["rolling_transaction_std"]
)

daily["frequency_time_zscore"] = (
    daily["frequency_time_zscore"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    daily[
        [
            "sender_id",
            "date",
            "daily_transaction_count",
            "rolling_transaction_mean",
            "rolling_transaction_std",
            "frequency_time_zscore"
        ]
    ].head(15)
)

   sender_id       date  daily_transaction_count  rolling_transaction_mean  \
0    U100000 2026-01-03                        1                       NaN   
1    U100000 2026-01-05                        1                       NaN   
2    U100000 2026-01-14                        1                       NaN   
3    U100000 2026-01-15                        1                  1.000000   
4    U100000 2026-01-16                        1                  1.000000   
5    U100000 2026-01-22                        1                  1.000000   
6    U100000 2026-01-24                        1                  1.000000   
7    U100000 2026-01-29                        1                  1.000000   
8    U100000 2026-01-31                        2                  1.000000   
9    U100000 2026-02-27                        1                  1.142857   
10   U100000 2026-03-02                        1                  1.142857   
11   U100000 2026-03-05                        1                

In [8]:
daily["time_series_anomaly_flag"] = (
    (daily["spending_time_zscore"] >= 3) |
    (daily["frequency_time_zscore"] >= 3)
).astype(int)

print(
    daily["time_series_anomaly_flag"].value_counts()
)

print("\nAnomaly Percentage:",
      daily["time_series_anomaly_flag"].mean() * 100)

time_series_anomaly_flag
0    12823
1      800
Name: count, dtype: int64

Anomaly Percentage: 5.8724216398737425


In [10]:
# anomaly score
# Time-series anomaly score
spending_score = (
    daily["spending_time_zscore"]
    .abs()
    .fillna(0)
    .clip(0, 5)
    / 5
) * 70

frequency_score = (
    daily["frequency_time_zscore"]
    .abs()
    .fillna(0)
    .clip(0, 5)
    / 5
) * 30

daily["time_series_anomaly_score"] = (
    spending_score + frequency_score
).round(2)

print(
    daily[
        [
            "sender_id",
            "date",
            "daily_spending",
            "daily_transaction_count",
            "spending_time_zscore",
            "frequency_time_zscore",
            "time_series_anomaly_score",
            "time_series_anomaly_flag"
        ]
    ]
    .sort_values("time_series_anomaly_score", ascending=False)
    .head(20)
)

      sender_id       date  daily_spending  daily_transaction_count  \
8283    U100616 2026-01-21        50616.03                        4   
1230    U100094 2026-03-10       122056.77                        3   
3702    U100285 2026-03-12        22596.98                        3   
7859    U100583 2026-03-29        96453.53                        3   
7033    U100527 2026-03-21       175473.03                        3   
5732    U100436 2026-03-12        46419.73                        3   
12809   U100945 2026-02-16        90132.40                        3   
7627    U100567 2026-03-02        19941.28                        3   
13339   U100981 2026-03-26       337501.30                        3   
6372    U100482 2026-01-20        16985.28                        3   
13388   U100984 2026-02-06        77598.24                        3   
1572    U100118 2026-03-08         5082.56                        3   
9308    U100689 2026-03-24       297518.24                        3   
4404  

In [11]:
# Daily anomaly results ko transaction-level data mein merge karo

daily_merge = daily[
    [
        "sender_id",
        "date",
        "rolling_spending_mean",
        "rolling_spending_std",
        "spending_time_zscore",
        "rolling_transaction_mean",
        "rolling_transaction_std",
        "frequency_time_zscore",
        "time_series_anomaly_score",
        "time_series_anomaly_flag"
    ]
].copy()

df["date"] = pd.to_datetime(df["timestamp"]).dt.normalize()

df = df.merge(
    daily_merge,
    on=["sender_id", "date"],
    how="left"
)

print("Final transaction-level shape:", df.shape)

print(
    df[
        [
            "transaction_id",
            "sender_id",
            "timestamp",
            "amount",
            "time_series_anomaly_score",
            "time_series_anomaly_flag"
        ]
    ].head(10)
)

Final transaction-level shape: (15000, 64)
  transaction_id sender_id           timestamp   amount  \
0      TX0000106   U100000 2026-01-03 13:26:04  2582.92   
1      TX0000129   U100000 2026-01-05 01:51:05  1327.52   
2      TX0000118   U100000 2026-01-14 06:21:25  1007.41   
3      TX0000045   U100000 2026-01-15 14:05:07   663.43   
4      TX0000130   U100000 2026-01-16 07:27:50   800.96   
5      TX0000121   U100000 2026-01-22 02:05:05  1036.17   
6      TX0000071   U100000 2026-01-24 05:30:30  1859.02   
7      TX0000012   U100000 2026-01-29 18:44:28   562.08   
8      TX0000132   U100000 2026-01-31 18:03:23   849.85   
9      TX0000102   U100000 2026-01-31 18:22:47  1173.37   

   time_series_anomaly_score  time_series_anomaly_flag  
0                       0.00                         0  
1                       0.00                         0  
2                       0.00                         0  
3                      16.41                         0  
4                     

In [12]:
print("\nTime-Series Flag:")
print(df["time_series_anomaly_flag"].value_counts())

print("\nMissing Time-Series Scores:")
print(df["time_series_anomaly_score"].isna().sum())


Time-Series Flag:
time_series_anomaly_flag
0    13890
1     1110
Name: count, dtype: int64

Missing Time-Series Scores:
0


In [14]:
# Save Phase 6 output

output_path = r"D:\Machine Learning\upi_fraud_detection\data\processed\time_series_anomaly_output.csv"

df.to_csv(output_path, index=False)

print("Phase 6 output saved successfully!")
print("File:", output_path)
print("Shape:", df.shape)

Phase 6 output saved successfully!
File: D:\Machine Learning\upi_fraud_detection\data\processed\time_series_anomaly_output.csv
Shape: (15000, 64)
